# REWIND pipeline: from videos to searchable captions, in one run

Runs the whole ingest pipeline on every video in `selection.csv` (about 1 h of MEVA + 1 h of UCA) and saves
everything to `outputs/batch/`. It shows **no pictures**: when it's done, open **`explore.ipynb`** and *Run All*.

**How to run:** check the config cell, then *Run All*. Everything runs top to bottom, in this order:

| Step | What it does | Code | Needs | Time for ~2 h of video (19-26 Nov runs) |
|---|---|---|---|---|
| Pre-flight | GPU visible? SigLIP's `sentencepiece`, YOLO, the Qwen folder there? Stops before anything runs if not | | | seconds |
| Fresh start | Deletes the old results in `outputs/batch/` (`FRESH_RUN`) | `batch.clean_outputs` | | seconds |
| 1. Motion gating + 2. Keyframes + GT check | Score 5 frames/s for motion, turn it into **segments**; pick keyframes; did they catch the labelled moments? | `motion_gating.py`, `keyframes.py` | CPU | ~8 min |
| 3. Events + 4. Hand-off | Inside segments: YOLO-World + ByteTrack, SigLIP, X-CLIP; pick the frames for captions and the index (`index_records.jsonl` + SigLIP embeddings) | `gpu_stages.py` | GPU | ~8 min (more with SigLIP) |
| 5a. Clip captions | Qwen-VL describes each **clip** (segment cut into ≤ 60 s, up to 6 frames in time order). A clip is what a search returns | `captions.py` | GPU, ~16 GB each | ~7 s per clip (÷ number of GPUs) |
| 5b. Frame captions | Qwen-VL describes each selected **frame** (people, clothes, vehicles, objects) | `captions.py` | GPU, ~16 GB each | ~5-10 s per frame (÷ number of GPUs): **hours** |

Each phase finishes every video before the next phase starts, so a problem late in the run never costs earlier
results. A video that fails is logged and skipped (its traceback: `error_<phase>.json` in its folder).

**If the run is interrupted** (VS Code disconnects, you stop a cell): set `FRESH_RUN = False` and *Run All* again.
Phases 1-2 skip finished videos; the caption phases continue at the next clip / frame. Everything the run prints
also goes to `outputs/batch/run_log.txt` with times (`tail -f outputs/batch/run_log.txt` in a terminal to follow it).

In [ ]:
import os
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

if Path("/mnt/5T_storage").exists():  # team VM: Hugging Face downloads go to the 5T disk, not the nearly full root disk
    os.environ.setdefault("HF_HOME", "/mnt/5T_storage/hf_cache")

import batch
import captions
import datasets as ds
import motion_gating as mg

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

## Config

In [ ]:
DATA_ROOT = Path(os.environ.get("REWIND_DATA_ROOT", "/mnt/5T_storage/clipsExtraction/datasets"))
OUT_ROOT = Path(os.environ.get("REWIND_OUT_ROOT", "outputs/batch"))   # one folder per video: <dataset>/<video>/
SELECTION_CSV = Path(os.environ.get("REWIND_SELECTION", "selection.csv"))  # reused if it exists; edit it to change videos
REMAKE_SELECTION = False   # True = pick the videos again (overwrites selection.csv)
MEVA_MINUTES = 60          # MEVA: whole 5-min time slots of all cameras until this many minutes (annotated slots first)
UCA_MINUTES = 60           # UCA: spread over categories (test split first) until this many minutes
UCA_VIDEO_DIR = "dev_videos"
MEVA_CAMERAS = None        # e.g. ["G328", "G336"]; None = every camera

FRESH_RUN = True           # True = delete everything in OUT_ROOT first. False = continue an interrupted run
KEEP_DECODE_CACHE = False  # True = keep stage 1's decode cache (saves ~8 min, but stage 1 times then read ~0)
STAGES = ["cpu", "gpu", "vlm_clips", "vlm_frames"]   # drop "vlm_frames" to skip the slow frame captions
FORCE = False              # True = redo cpu/gpu even for videos that already have results (FRESH_RUN does this anyway)
MOCK_MODELS = os.environ.get("REWIND_MOCK", "") == "1"   # fake GPU models + captioner: plumbing check only

# Qwen2.5-VL-7B-Instruct, already downloaded on the VM (any local folder with config.json, or a Hugging Face name)
VLM_MODEL = os.environ.get("REWIND_VLM_MODEL", "/mnt/5T_storage/hat_detection_vlm/hf_cache/hub/"
                           "models--Qwen--Qwen2.5-VL-7B-Instruct/snapshots/cc594898137f460bfe9f0759e9844b3ce807cfb5")
VLM_DEVICE = ["cuda:0", "cuda:1"]   # GPUs to caption on: a list = one copy of the VLM per GPU, each captions about half
                                    # the videos (~2x faster). One GPU: "cuda:0" (or "cuda:1" if the other group uses 0)

CFG = {
    "out_root": OUT_ROOT,
    "gate": "mog2",        # gate that feeds keyframes and stages 3-4: "mog2" | "diff" | "fused" | "deep"
    "deep_gate": False,    # DINOv2 gate = a GPU pass over every whole video; docs/pipeline.md found it not worth it
    "scan": dict(          # stage 1, slow pass (changing these re-decodes the videos)
        sample_fps=5.0, width=320, blur_ksize=5, diff_threshold=15,
        mog2_history=500, mog2_var_threshold=16.0, mog2_learning_rate=-1.0,
        mog2_illum_ncc=0.7, mog2_color=True, morph_kernel=3, ignore_polygons=[], hash_grid=4, hist_bins=32,
    ),
    "segments": dict(      # stage 1, cheap
        gates=["diff", "mog2"], threshold_mode="adaptive", k=3.0, q_noise=0.5,
        floor={"diff": 0.001, "mog2": 0.001, "deep": 0.004}, fixed_threshold={"mog2": 0.004, "diff": 0.006},
        stop_ratio=0.5, smooth_s=1.0, merge_gap_s=3.0, min_duration_s=1.0, pad_s=2.0,
    ),
    "keyframes": dict(     # stage 2, cheap
        metric="phash", change_threshold=6.0, dedup_threshold=2.5, min_gap_s=3.0, max_gap_s=30.0, heartbeat_s=300.0,
    ),
    "v2": {                # stages 3-4 (all keys: gpu_stages.DEFAULT_CONFIG)
        "sampling": {"primary": "kcenter", "export_jpgs": ["kcenter"]},
        "runtime": {"device": "auto", "mock_models": MOCK_MODELS},
    },
    "vlm": {               # stage 5 captions (all keys: captions.CAPTION_CONFIG)
        "model": VLM_MODEL, "device": VLM_DEVICE,
        "source": "auto",                # frames to caption: "index" = stage 4, "keyframes" = stage 2, "auto" = index if present
        "clip_max_s": 60.0,              # segments are cut into clips of at most this length
        "clip_max_frames": 6,            # frames the VLM sees per clip
        "batch_size": 8,                 # frames per GPU call in 5b (more = faster per frame); 4 if CUDA runs out of memory
        "mock": MOCK_MODELS,
    },
}

## Pre-flight check
Everything the run needs, checked before anything is deleted or started: data, GPU (via PyTorch, so it works even
when `nvidia-smi` is missing), the packages the GPU models need, and the Qwen folder. A ❌ on something a selected
stage needs stops here with the fix; ⚠️ is informational.

In [ ]:
import importlib
import shutil


def _ver(mod):
    """(installed?, version or error)"""
    try:
        return True, str(getattr(importlib.import_module(mod), "__version__", "installed"))
    except Exception as e:  # missing or broken
        return False, f"missing ({type(e).__name__})"


checks = []   # (what, ok, detail, needed_by)
checks.append(("data folder", DATA_ROOT.exists() or MOCK_MODELS, str(DATA_ROOT), {"cpu"}))
checks.append(("OpenCV (cv2)", *_ver("cv2"), {"cpu"}))
free_gb = shutil.disk_usage(OUT_ROOT if OUT_ROOT.exists() else Path(".")).free / 1e9
checks.append(("free disk for outputs", free_gb > 20, f"{free_gb:.0f} GB", {"cpu"}))
t_ok, tv = _ver("torch")
checks.append(("PyTorch", t_ok, tv, {"gpu", "vlm_clips", "vlm_frames"}))
if t_ok:
    import torch
    n_gpu = torch.cuda.device_count() if torch.cuda.is_available() else 0
    checks.append(("GPU visible to PyTorch", n_gpu > 0 or MOCK_MODELS, f"{n_gpu} GPU(s)", {"gpu", "vlm_clips", "vlm_frames"}))
    for i in range(n_gpu):
        free, total = torch.cuda.mem_get_info(i)
        checks.append((f"  cuda:{i} {torch.cuda.get_device_name(i)}", True,
                       f"{free / 1e9:.1f} of {total / 1e9:.1f} GB free", set()))
    for dev in (VLM_DEVICE if isinstance(VLM_DEVICE, (list, tuple)) else [VLM_DEVICE]):
        if not str(dev).startswith("cuda"):
            continue
        k = int(dev.split(":")[1]) if ":" in dev else 0
        ok_dev = k < n_gpu and torch.cuda.mem_get_info(k)[0] / 1e9 > 17
        checks.append((f"{dev} has room for the VLM (~17 GB)", ok_dev or MOCK_MODELS,
                       "see free memory above; in use by the other group? remove it from VLM_DEVICE",
                       {"vlm_clips", "vlm_frames"}))
checks.append(("transformers", *_ver("transformers"), {"gpu", "vlm_clips", "vlm_frames"}))
checks.append(("sentencepiece (SigLIP tokenizer; missing = no search embeddings)", *_ver("sentencepiece"), {"gpu"}))
checks.append(("ultralytics (YOLO-World)", *_ver("ultralytics"), {"gpu"}))
mp = Path(VLM_MODEL)
vlm_ok = MOCK_MODELS or (mp / "config.json").exists() or (not mp.is_absolute() and "/" in VLM_MODEL)
checks.append(("VLM model folder (config.json)", vlm_ok, VLM_MODEL if not MOCK_MODELS else "mock", {"vlm_clips", "vlm_frames"}))
if t_ok and not MOCK_MODELS:
    has_tf, v = _ver("transformers")
    from packaging import version
    new_enough = has_tf and version.parse(v) >= version.parse("4.49")
    checks.append(("transformers >= 4.49 (Qwen2.5-VL)", new_enough, v, {"vlm_clips", "vlm_frames"}))

stop = []
for what, ok_, detail, need in checks:
    needed = bool(need & set(STAGES)) and not (MOCK_MODELS and need <= {"gpu", "vlm_clips", "vlm_frames"})
    mark = "✅" if ok_ else ("❌" if needed else "⚠️")
    print(f"{mark} {what}: {detail}")
    if not ok_ and needed:
        stop.append(what)
if stop:
    raise SystemExit("Fix before running: " + "; ".join(stop) + "\n(sentencepiece / ultralytics: pip install in this "
                     "kernel's env and restart the kernel; VLM folder: VLM_MODEL in the config cell; GPU: VLM_DEVICE)")
print("\nall set" + ("  (MOCK MODELS: numbers are meaningless)" if MOCK_MODELS else ""))

## 1. What is on disk
All videos of both datasets, before picking. Durations come from the file names (MEVA) and the annotations (UCA).

In [ ]:
meva_inv = ds.meva_inventory(DATA_ROOT)
uca_inv = ds.uca_inventory(DATA_ROOT, UCA_VIDEO_DIR)
inv = pd.concat([meva_inv, uca_inv], ignore_index=True)
display(ds.describe(inv))
if len(meva_inv):
    print("MEVA cameras:", meva_inv.camera_id.value_counts().to_dict(),
          "| clips shorter than 4.5 min:", int((meva_inv.duration_s < 270).sum()),
          "| annotation matches:", meva_inv.gt_match.value_counts().to_dict())
if len(uca_inv):
    print("UCA splits:", uca_inv.split.replace("", "(no annotation)").value_counts().to_dict())
    print("UCA categories:", uca_inv.category.value_counts().to_dict())

## 2. Which videos
`selection.csv` lists them (path, camera, wall-clock start, ground-truth file). Edit it by hand to change the set.

In [ ]:
sel = ds.load_or_make_selection(SELECTION_CSV, DATA_ROOT, remake=REMAKE_SELECTION, meva_min=MEVA_MINUTES,
                                uca_min=UCA_MINUTES, uca_video_dir=UCA_VIDEO_DIR, meva_cameras=MEVA_CAMERAS)
missing = [p for p in sel.path if not Path(p).exists()]
if missing:
    print(f"WARNING: {len(missing)} files in {SELECTION_CSV} don't exist, e.g. {missing[0]}")
display(sel.assign(minutes=sel.duration_s.astype(float) / 60).groupby("dataset").agg(
    videos=("video_id", "size"), minutes=("minutes", "sum"),
    with_ground_truth=("gt_match", lambda s: int((s != "none").sum()))).round(1))
display(sel[["dataset", "video_id", "camera_id", "start_wallclock", "duration_s", "category", "split", "gt_match", "n_gt"]])

## 3. Fresh start
Deletes the previous results (`FRESH_RUN = True`), then starts a new log at `outputs/batch/run_log.txt`.

In [ ]:
if FRESH_RUN:
    batch.clean_outputs(OUT_ROOT, keep_decode_cache=KEEP_DECODE_CACHE)
else:
    print("FRESH_RUN = False: keeping existing results, continuing where the last run stopped")
OUT_ROOT.mkdir(parents=True, exist_ok=True)
log = batch.make_logger(OUT_ROOT / "run_log.txt")
log(f"run started, phases {STAGES}, fresh={FRESH_RUN}, {len(sel)} videos")
PHASE_MIN = {}


def run_phase(name, **kw):
    """One phase over every video; returns the summary table. Its wall time goes into PHASE_MIN."""
    if name not in STAGES:
        print(f"'{name}' not in STAGES, skipped")
        return pd.read_csv(OUT_ROOT / "batch_summary.csv") if (OUT_ROOT / "batch_summary.csv").exists() else None
    t0 = time.perf_counter()
    s = batch.run_batch(sel, CFG, stages=[name], force=FORCE, log=log, **kw)
    PHASE_MIN[name] = (time.perf_counter() - t0) / 60
    return s

## 4. Phase 1 (CPU): motion gating, keyframes, ground-truth check
One line per video: share of the video active, keyframes, labelled moments with a keyframe.

In [ ]:
summary = run_phase("cpu")

## 5. Phase 2 (GPU): events and the frames for captions and search
Stage 3 (YOLO-World + ByteTrack, SigLIP, X-CLIP) and stage 4 (which frames go to the captioner and the index).
The check underneath flags the problem of the 19 Nov run: a model that didn't load is skipped silently, and
without SigLIP there are no search embeddings.

In [ ]:
summary = run_phase("gpu")
if summary is not None and "status_gpu" in summary:
    g = summary[summary.status_gpu.astype(str) == "ok"]
    if len(g):
        skipped = sorted({x for v in g.get("skipped_models", pd.Series(dtype=str)).dropna().astype(str) for x in [v] if x})
        no_emb = int((g.embedding_dim.fillna(0) == 0).sum()) if "embedding_dim" in g else len(g)
        print(f"{len(g)} videos with GPU results; videos without SigLIP embeddings: {no_emb}")
        if skipped:
            print("❌ models skipped:", skipped[0][:200])
        if no_emb:
            print("❌ no search embeddings: install sentencepiece + protobuf in this env, restart the kernel, "
                  "set FRESH_RUN = False and STAGES = ['gpu', 'vlm_clips', 'vlm_frames'], Run All")

## 6. Phase 3a (GPU): clip captions
The phase-2 models are freed first. With one GPU the VLM is loaded once here (about a minute) and reused in 3b;
with a list of GPUs in `VLM_DEVICE`, each GPU gets its own copy and about half the videos (lines start with `[cuda:0]` / `[cuda:1]`).
Every clip gets a JSON description: summary, actions, people, vehicles, objects, anything unusual, what was unclear.
The VLM sees the clip's frames in time order, labelled with their offsets ("+12.3 s").

In [ ]:
batch.free_gpu_memory()
captioner = captions.Captioner(CFG["vlm"])   # one GPU: loads on first use, shared by 3a and 3b
# several GPUs (VLM_DEVICE is a list): each phase starts one worker per GPU, each loading its own VLM (~1 min)
summary = run_phase("vlm_clips", captioner=captioner)

## 7. Phase 3b (GPU): frame captions (slow)
One caption per selected frame, `batch_size` frames per GPU call. This is the longest step: at ~5-10 s per frame,
1,000 frames take 1.5-3 h. Stop it any time: what's done is saved, and with `FRESH_RUN = False` a new *Run All*
continues at the next frame. Remove `"vlm_frames"` from `STAGES` to skip it.

In [ ]:
if "vlm_frames" in STAGES and summary is not None:
    n_fr = 0
    for _, r in sel.iterrows():
        try:
            n_fr += len(captions.load_frames_to_caption(batch.out_dir(batch.merged_cfg(CFG), r), CFG["vlm"]["source"])[0])
        except FileNotFoundError:
            pass
    print(f"{n_fr} frames to caption (minus any done before)")
summary = run_phase("vlm_frames", captioner=captioner)

## 8. Summary
Per dataset. Pictures, captions next to their frames, and failure examples: **`explore.ipynb`**.
Caption quality (detector cross-check, comparison with the labels, the hand check): `explore.ipynb` section 6.

In [ ]:
s = pd.read_csv(OUT_ROOT / "batch_summary.csv")
ok = s[s.get("status_cpu", pd.Series(dtype=str)).astype(str) == "ok"] if "status_cpu" in s else s.iloc[0:0]
if len(ok):
    g = ok.groupby("dataset")
    tab = pd.DataFrame({
        "videos": g.size(),
        "hours": g.duration_s.sum() / 3600,
        "active %": g.apply(lambda d: (d.active_pct * d.duration_s).sum() / d.duration_s.sum()),
        "keyframes": g.keyframes.sum(),
        "reduction (x fewer than 1 fps)": g.apply(lambda d: d.naive_frames.sum() / max(1, d.keyframes.sum())),
    })
    if "gt_kf_hit" in ok:
        w = ok.dropna(subset=["gt_kf_hit"])
        if len(w):
            gw = w.groupby("dataset")
            tab["GT items"] = gw.gt_items.sum()
            tab["GT items with a keyframe %"] = gw.apply(lambda d: 100 * (d.gt_kf_hit * d.gt_items).sum() / d.gt_items.sum())
            tab["same, uniform sampling %"] = gw.apply(lambda d: 100 * (d.gt_kf_hit_uniform * d.gt_items).sum() / d.gt_items.sum())
    if "index_records" in ok:
        tab["index records"] = g.index_records.sum()
        tab["with SigLIP embeddings"] = g.apply(lambda d: int((d.embedding_dim.fillna(0) > 0).sum()))
    if "vlm_clips" in ok:
        tab["clip captions"] = g.vlm_clips.sum()
        tab["frame captions"] = g.vlm_frames.sum()
        tab["unparsed"] = g.vlm_parse_failed.sum()
        tab["s per clip"] = g.vlm_s_per_clip.mean()
        tab["s per frame"] = g.vlm_s_per_frame.mean()
    display(tab.round(2))
errs = s[[c for c in s.columns if c.startswith("status_")]].astype(str).apply(lambda r: r.str.startswith("error").any(), axis=1)
if errs.any():
    display(Markdown(f"**{int(errs.sum())} video(s) with errors** (traceback in `error_<phase>.json` in their folder):"))
    display(s.loc[errs, ["dataset", "video_id"] + [c for c in s.columns if c.startswith("status_")]])
if PHASE_MIN:
    display(pd.Series(PHASE_MIN, name="minutes").round(1).to_frame())
log(f"run finished: {', '.join(f'{k} {v:.1f} min' for k, v in PHASE_MIN.items())}")
print(f"summary table: {OUT_ROOT / 'batch_summary.csv'}  ·  log: {OUT_ROOT / 'run_log.txt'}  ·  next: explore.ipynb")